In [1]:
import os, sys, json, time, hashlib, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import pyarrow, sklearn, mlflow
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

# ---- config ----
EXPERIMENT = "rain_v2_3month_lgbm_cv"
DATA_PATH = "../data/processed/rain_v2_3month_dataset_02.parquet"
FEATURE_PATH = "../data/processed/rain_v2_3month_feature_list_02.json"
PROCESSED = Path(DATA_PATH).parent
TRAIN_START, TRAIN_END = 1901, 2002
VAL_START, VAL_END = 2003, 2013
TEST_START, TEST_END = 2014, 2025   # test is NOT touched in this notebook
OUTLIER_THRESHOLD = 6
SD_FLOOR = 5
RANDOM_STATE = 42
BUFFER_FROM_MONTH = 10              # origin Oct-Dec before a val block are dropped from train

FOLDS = {
    "A": dict(train_start=1901, train_end=1950, val_start=1951, val_end=1975),
    "B": dict(train_start=1901, train_end=1975, val_start=1976, val_end=2002),
}

DEFAULT_PARAMS = dict(
    n_estimators=300, learning_rate=0.05, num_leaves=31, max_depth=-1,
    min_child_samples=100, subsample=0.8, colsample_bytree=0.8,
    reg_lambda=1.0, random_state=RANDOM_STATE,
)

def file_md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
DATASET_MD5 = file_md5(DATA_PATH)

for name, ver in [("python", sys.version.split()[0]), ("lightgbm", lgb.__version__),
                  ("numpy", np.__version__), ("pandas", pd.__version__),
                  ("pyarrow", pyarrow.__version__), ("scikit-learn", sklearn.__version__),
                  ("mlflow", mlflow.__version__)]:
    print(f"{name:13s}{ver}")
print("git commit  ", GIT_COMMIT)
print("dataset md5 ", DATASET_MD5)

# ---- MLflow (token is read from .env, never printed) ----
load_dotenv("../../.env")
os.environ["MLFLOW_TRACKING_USERNAME"] = "Sharanya-0224"
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.environ["DAGSHUB_USER_TOKEN"]
mlflow.set_tracking_uri("https://dagshub.com/Sharanya-0224/Climate-Change-Awareness-Dashboard.mlflow")
mlflow.set_system_metrics_sampling_interval(1)
mlflow.set_experiment(EXPERIMENT)
print("MLflow experiment:", EXPERIMENT)

/home/sharanya/Climate-Change-Awareness-Dashboard/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


python       3.12.3
lightgbm     4.7.0
numpy        2.0.2
pandas       2.3.3
pyarrow      25.0.1
scikit-learn 1.9.1
mlflow       3.17.1
git commit   28b6139
dataset md5  8ea8f7473d958fb34548ab9e66dbff97
MLflow experiment: rain_v2_3month_lgbm_cv


In [2]:
df = pd.read_parquet(DATA_PATH)
with open(FEATURE_PATH) as f:
    FEATURE_INFO = json.load(f)
FEATURES = FEATURE_INFO["inputs"]

REQUIRED = FEATURE_INFO["id_columns"] + [FEATURE_INFO["target"]] + FEATURE_INFO["target_extra"]
missing = [c for c in REQUIRED + FEATURES if c not in df.columns]
assert not missing, f"Missing columns: {missing}\nColumns are: {list(df.columns)}"
assert len(FEATURES) == 23, len(FEATURES)
assert not df[FEATURES].isna().any().any(), "NaN in inputs"

print("shape:", df.shape)
print("points:", df["key"].nunique(), "(file says", FEATURE_INFO["params"]["points"], ")")
for name, a, b in [("train", TRAIN_START, TRAIN_END), ("val", VAL_START, VAL_END),
                   ("test", TEST_START, TEST_END)]:
    part = df[df["year"].between(a, b)]
    print(f"{name:5s} rows {len(part):>9,}  years {part['year'].min()}-{part['year'].max()}")
print("regions:", sorted(df["region"].unique()))
print("split column:", df["split"].value_counts().to_dict())

shape: (7306839, 34)
points: 4927 (file says 4927 )
train rows 5,980,344  years 1901-2002
val   rows   632,499  years 2003-2013
test  rows   693,996  years 2014-2025
regions: ['central', 'east_northeast', 'northwest', 'south_peninsula']
split column: {'train': 5980344, 'test': 693996, 'val': 632499}


In [3]:
#reusable functions
SEASONS = {12: "DJF", 1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM",
           6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"}

def get_season(month):
    return month.map(SEASONS)   # by ORIGIN month

def mse_skill(y, p):
    # skill vs climatology (climatology predicts z = 0)
    y = np.asarray(y); p = np.asarray(p)
    return 1.0 - np.sum((y - p) ** 2) / np.sum(y ** 2)

def evaluate(part, pred, drop_extreme=False):
    """Overall, by origin-month season, by region. Data is an argument."""
    d = part[["month", "region", "target_z"]].copy()
    d["pred"] = np.asarray(pred)
    if drop_extreme:
        d = d[d["target_z"].abs() <= OUTLIER_THRESHOLD]
    out = {"overall": mse_skill(d["target_z"], d["pred"])}
    for s, g in d.groupby(get_season(d["month"])):
        out[f"season_{s}"] = mse_skill(g["target_z"], g["pred"])
    for r, g in d.groupby("region"):
        out[f"region_{r}"] = mse_skill(g["target_z"], g["pred"])
    return out

def make_split(data, train_start, train_end, val_start, val_end, buffer=True):
    """Train and val by origin year. Buffer drops train origins Oct-Dec of
    the year before val_start (their target crosses into the val block)."""
    tr = data[data["year"].between(train_start, train_end)]
    va = data[data["year"].between(val_start, val_end)]
    if buffer:
        cross = (tr["year"] == val_start - 1) & (tr["month"] >= BUFFER_FROM_MONTH)
        tr = tr[~cross]
    return tr, va

def fit_lgbm(X, y, params):
    model = lgb.LGBMRegressor(**params)
    model.fit(X, y)
    return model

def run_cv_setting(run_name, data, features, changes=None, clip=None, drop_cols=()):
    """One MLflow run = one setting, both folds. Scored on the UNCLIPPED target."""
    params = {**DEFAULT_PARAMS, **(changes or {})}
    feats = [c for c in features if c not in drop_cols]
    fold_skill, pooled_val, pooled_pred = {}, [], []

    with mlflow.start_run(run_name=run_name, log_system_metrics=False):
        mlflow.log_params(params)
        mlflow.log_params({"clip_target_z": clip, "dropped_features": ",".join(drop_cols) or "none",
                           "n_features": len(feats), "sd_floor": SD_FLOOR})
        mlflow.set_tags({"git_commit": GIT_COMMIT, "dataset_md5": DATASET_MD5,
                         "split_rule": "rolling CV A=1901-50/1951-75, B=1901-75/1976-2002, buffer Oct-Dec",
                         "known_leak": "normals use all train years"})
        for fold, cfg in FOLDS.items():
            tr, va = make_split(data, **cfg)
            y_tr = tr["target_z"].clip(-clip, clip) if clip else tr["target_z"]
            t0 = time.time()
            model = fit_lgbm(tr[feats], y_tr, params)
            fit_s = time.time() - t0
            pred = model.predict(va[feats])
            res = evaluate(va, pred)
            fold_skill[fold] = res["overall"]
            mlflow.log_metric(f"fold{fold}_fit_seconds", fit_s)
            for k, v in res.items():
                mlflow.log_metric(f"fold{fold}_{k}", v)
            pooled_val.append(va); pooled_pred.append(pred)
            print(f"{run_name} fold {fold}: skill {res['overall']:.4f}  ({fit_s:.0f}s)")
        cv_mean = float(np.mean(list(fold_skill.values())))
        mlflow.log_metric("cv_mean_skill", cv_mean)
        mlflow.log_metric("cv_both_folds_positive", int(all(v > 0 for v in fold_skill.values())))
        for k, v in evaluate(pd.concat(pooled_val), np.concatenate(pooled_pred)).items():
            mlflow.log_metric(f"cv_pooled_{k}", v)
    return {"run": run_name, "cv_mean": cv_mean, **{f"fold{k}": v for k, v in fold_skill.items()}}

In [5]:
#two CV fits
RESULTS = []

res = run_cv_setting(
    run_name="cv_run1_default",
    data=df,
    features=FEATURES,
)
RESULTS.append(res)
print(pd.DataFrame(RESULTS).round(4))

2026/11/12 11:37:59 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/11/12 11:37:59 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.117997 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3964
[LightGBM] [Info] Number of data points in the train set: 2910772, number of used features: 23
[LightGBM] [Info] Start training from score 0.017564
cv_run1_default fold A: skill -0.0448  (47s)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.178333 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4033
[LightGBM] [Info] Number of data points in the train set: 4385016, number of used features: 23
[LightGBM] [Info] Start training from score -0.003742
cv_run1_default fold B: skill 0.0052  (63s)
🏃 View run cv_run1_default at: https://dagshub.com/Sharanya-0224/Climate-Change-Awareness-Dashboard.mlflow/#/experiments/11/runs/58702410d3734961864c2920ae2c1c3d
🧪 View experiment at: https://dagshub.com/Sharanya-0224/Clim

2026/11/12 11:42:31 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/11/12 11:42:32 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


               run  cv_mean   foldA   foldB
0  cv_run1_default  -0.0198 -0.0448  0.0052


hyper parameter tuning: changing values in the model to check for better outputs

In [7]:
res = run_cv_setting(
    run_name="cv_run2_leaves15",
    data=df,
    features=FEATURES,
    changes={"num_leaves": 15},
)
RESULTS.append(res)
print(pd.DataFrame(RESULTS).round(4))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.048078 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3964
[LightGBM] [Info] Number of data points in the train set: 2910772, number of used features: 23
[LightGBM] [Info] Start training from score 0.017564
cv_run2_leaves15 fold A: skill -0.0500  (37s)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.123374 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4033
[LightGBM] [Info] Number of data points in the train set: 4385016, number of used features: 23
[LightGBM] [Info] Start training from score -0.003742
cv_run2_leaves15 fold B: skill 0.0063  (52s)
🏃 View run cv_run2_leaves15 at: https://dagshub.com/Sharanya-0224/Climate-Change-Awareness-Dashboard.mlflow/#/experiments/11/runs/3ad8d8adbc71492186b420b652

Rolling CV: runs 1 and 2 (default and num_leaves 15)

Setup: two folds, A = train 1901-1950 / validate 1951-1975, B = train 1901-1975 / validate 1976-2002.
Skill = MSE skill vs climatology on the unclipped target_z. Season = origin month.

| Run | Setting | Fold A | Fold B | CV mean |
|---|---|---|---|---|
| 1 | default (31 leaves) | -4.48% | +0.52% | -1.98% |
| 2 | num_leaves 15 | -5.00% | +0.63% | -2.19% |

What this tells us:

1. **Tree size is not the problem.** Run 2 differs from run 1 by 0.5 points in fold A and 0.1 in fold B,
   in opposite directions. This is within noise, so we cannot say fewer leaves helps or hurts.
2. **In fold A the model is worse than climatology.** A negative skill means that predicting z = 0
   would have scored better. The model learned patterns from 1901-1950 that did not hold in 1951-1975.
3. **The signal is weak and not stable across periods.** Skill goes from about -4.5% (50 train years)
   to about +0.5% (75 train years) to 3.0% on val 2003-2013 (102 train years, default model).
   More training years and a later validation period go together with better skill.
4. **We cannot yet say why.** Possible causes: too little training data in fold A, lower station
   density in the early years (not checked), the ENSO and IOD links changing over time, and few
   independent years (about 25 per fold, because all points share the same climate state). None is tested.

In [8]:
res = run_cv_setting(
    run_name="cv_run3_minchild3000",
    data=df,
    features=FEATURES,
    changes={"min_child_samples": 3000},
)
RESULTS.append(res)
print(pd.DataFrame(RESULTS).round(4))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.100024 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3964
[LightGBM] [Info] Number of data points in the train set: 2910772, number of used features: 23
[LightGBM] [Info] Start training from score 0.017564
cv_run3_minchild3000 fold A: skill -0.0484  (52s)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.170308 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4033
[LightGBM] [Info] Number of data points in the train set: 4385016, number of used features: 23
[LightGBM] [Info] Start training from score -0.003742
cv_run3_minchild3000 fold B: skill 0.0092  (56s)
🏃 View run cv_run3_minchild3000 at: https://dagshub.com/Sharanya-0224/Climate-Change-Awareness-Dashboard.mlflow/#/experiments/11/runs/8455edba0cb04f209fa08f65710cd331
🧪 View experiment at: https://dagshub.com/Sha

In [9]:
#both changes together
res = run_cv_setting(
    run_name="cv_run4_leaves15_minchild3000",
    data=df,
    features=FEATURES,
    changes={"num_leaves": 15, "min_child_samples": 3000},
)
RESULTS.append(res)
print(pd.DataFrame(RESULTS).round(4))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.084361 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3964
[LightGBM] [Info] Number of data points in the train set: 2910772, number of used features: 23
[LightGBM] [Info] Start training from score 0.017564
cv_run4_leaves15_minchild3000 fold A: skill -0.0390  (20s)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.114839 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4033
[LightGBM] [Info] Number of data points in the train set: 4385016, number of used features: 23
[LightGBM] [Info] Start training from score -0.003742
cv_run4_leaves15_minchild3000 fold B: skill 0.0027  (28s)
🏃 View run cv_run4_leaves15_minchild3000 at: https://dagshub.com/Sharanya-0224/Climate-Change-Awareness-Dashboard.mlflow/#/experiments/11/runs/20eb41a90e384a238bcdc3fa6caaf8e6
🧪 View experiment 

Above run (num_leaves 15 and min_child_samples 3000)
Fold A -3.90%, fold B +0.27%, CV mean -1.81% (default -1.98%).
Inference: the two complexity limits together help a little more than each alone (fold A
improves by 0.6 points), but fold B drops by 0.25 and the mean gain is 0.17 points, below the
0.3 point rule. Across runs 1 to 4 the CV mean stays between -1.8% and -2.2%, and fold A stays
below 0. Model complexity is therefore not the main cause of the weak CV skill. Runs 5 (target
clip) and 6 (no lat/lon) test other causes.

three runs with different complexity give almost the same fold A, between -4.5% and -5.0%. So the model does not fail because it memorises too much. The problem is in the data or the period: only 50 train years, a different climate state in 1951-1975, and possibly early station density. Hyperparameters cannot fix that. 

The next runs attack different causes (extreme rows, location memorising), so they are more useful now.

In [10]:
# Run 5: target_z clipped at +-5 in training only, scored on the unclipped target
RESULTS.append(run_cv_setting(
    run_name="cv_run5_clip5",
    data=df, features=FEATURES, clip=5,
))

# Run 6: without lat and lon (region one-hots stay)
RESULTS.append(run_cv_setting(
    run_name="cv_run6_no_latlon",
    data=df, features=FEATURES, drop_cols=("lat", "lon"),
))

res_df = pd.DataFrame(RESULTS)
res_df["both_folds_above_0"] = (res_df["foldA"] > 0) & (res_df["foldB"] > 0)
print(res_df.round(4))

# Selection rule (fixed before the runs)
default_mean = res_df.loc[res_df["run"] == "cv_run1_default", "cv_mean"].iloc[0]
passing = res_df[res_df["both_folds_above_0"]]
if passing.empty:
    print("\nNo setting has both folds above 0. No winner by the rule. Keep the default.")
else:
    best = passing.sort_values("cv_mean", ascending=False).iloc[0]
    gain = (best["cv_mean"] - default_mean) * 100
    print(f"\nBest passing: {best['run']}  cv_mean {best['cv_mean']:.4f}  gain vs default {gain:.2f} points")
    print("Keep default (gain below 0.3 points)." if gain < 0.3 else "Choose this setting.")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.067084 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3964
[LightGBM] [Info] Number of data points in the train set: 2910772, number of used features: 23
[LightGBM] [Info] Start training from score 0.016278
cv_run5_clip5 fold A: skill -0.0524  (24s)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.108664 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4033
[LightGBM] [Info] Number of data points in the train set: 4385016, number of used features: 23
[LightGBM] [Info] Start training from score -0.005427
cv_run5_clip5 fold B: skill 0.0066  (33s)
🏃 View run cv_run5_clip5 at: https://dagshub.com/Sharanya-0224/Climate-Change-Awareness-Dashboard.mlflow/#/experiments/11/runs/09320d6bbb244b7a8ccd73a88757a3b5
🧪 View experiment at: https://dagshub.com/Sharanya-0224/Climate-Ch

Rolling CV: result and selection
| Run | Setting | Fold A | Fold B | CV mean |
|---|---|---|---|---|
| 1 | default | -4.48% | +0.52% | -1.98% |
| 2 | num_leaves 15 | -5.00% | +0.63% | -2.19% |
| 3 | min_child_samples 3000 | -4.84% | +0.92% | -1.96% |
| 4 | both | -3.90% | +0.27% | -1.81% |
| 5 | target clip +-5 | -5.24% | +0.66% | -2.29% |
| 6 | no lat/lon | -4.93% | +0.37% | -2.28% |

Decision: no setting has both folds above 0, so the rule gives no winner. We keep the default.

Inference: 
- six different settings give the same picture (fold A between -3.9% and -5.2%,
fold B between +0.3% and +0.9%). 
- Model complexity, extreme rows and location memorising are
not the cause of the weak CV skill. 
- The cause is more likely the data and the period: few
training years in the early folds, a changing climate state, and early station density
- The val skill of 3.01% (2003-2013) is a modest result for recent years. 
- It is not proof of stable skill in older periods.

In [12]:
import sys, lightgbm
print(sys.executable)
print(lightgbm.__version__, lightgbm.__file__)

/home/sharanya/Climate-Change-Awareness-Dashboard/venv/bin/python
4.7.0 /home/sharanya/Climate-Change-Awareness-Dashboard/venv/lib/python3.12/site-packages/lightgbm/__init__.py


In [4]:
import mlflow.lightgbm

PKG_DIR = Path("../inference/model_package")
PKG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_TXT = PKG_DIR / "model.txt"

train = df[df["year"].between(TRAIN_START, TRAIN_END)]
val = df[df["year"].between(VAL_START, VAL_END)]

freeze = {
    "params": DEFAULT_PARAMS, "clip_target_z": None, "features": FEATURES,
    "dataset_md5": DATASET_MD5, "git_commit": GIT_COMMIT, "sd_floor_mm": SD_FLOOR,
    "lightgbm_version": lgb.__version__, "train_years": [TRAIN_START, TRAIN_END],
    "selection": "default kept: no CV setting had both folds above 0",
}
with open(PKG_DIR / "freeze_record.json", "w") as f:
    json.dump(freeze, f, indent=2)

mlflow.set_experiment("rain_v2_3month_lightgbm")
with mlflow.start_run(run_name="rain_v2_lgbm_final", log_system_metrics=False):
    mlflow.log_params(DEFAULT_PARAMS)
    mlflow.set_tags({"git_commit": GIT_COMMIT, "dataset_md5": DATASET_MD5,
                     "lightgbm_version": lgb.__version__,
                     "season_definition": "origin month",
                     "known_leak": "normals use all train years"})

    t0 = time.time()
    model = fit_lgbm(train[FEATURES], train["target_z"], DEFAULT_PARAMS)
    mlflow.log_metric("fit_seconds", time.time() - t0)

    pred_val = model.predict(val[FEATURES])
    res_all = evaluate(val, pred_val)
    res_noext = evaluate(val, pred_val, drop_extreme=True)
    for k, v in res_all.items():
        mlflow.log_metric(f"val_{k}", v)
    for k, v in res_noext.items():
        mlflow.log_metric(f"val_noext_{k}", v)
    pred_tr = model.predict(train[FEATURES])
    mlflow.log_metric("train_overall", mse_skill(train["target_z"], pred_tr))

    # save booster and reload check
    model.booster_.save_model(str(MODEL_TXT))
    booster = lgb.Booster(model_file=str(MODEL_TXT))
    reload_diff = float(np.max(np.abs(booster.predict(val[FEATURES]) - pred_val)))
    mlflow.log_metric("reload_max_abs_diff", reload_diff)

    mlflow.log_artifact(str(MODEL_TXT))
    mlflow.log_artifact(str(PKG_DIR / "freeze_record.json"))
    mlflow.lightgbm.log_model(model.booster_, name="model")

print(f"val skill all rows : {res_all['overall']:.4f}   (4.7.0 run: 0.0301)")
print(f"val skill no |z|>6 : {res_noext['overall']:.4f}   (4.7.0 run: 0.0335)")
print(f"train skill        : {mse_skill(train['target_z'], pred_tr):.4f}")
print(f"reload max abs diff: {reload_diff:.2e}   (must be below 1e-9)")
print("model.txt size MB  :", round(MODEL_TXT.stat().st_size / 1e6, 2))
print({k: round(v, 4) for k, v in res_all.items()})

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.165933 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4046
[LightGBM] [Info] Number of data points in the train set: 5980344, number of used features: 23
[LightGBM] [Info] Start training from score 0.000904
🏃 View run rain_v2_lgbm_final at: https://dagshub.com/Sharanya-0224/Climate-Change-Awareness-Dashboard.mlflow/#/experiments/10/runs/cc32f83f9f2b4aab92d7e8da713427d7
🧪 View experiment at: https://dagshub.com/Sharanya-0224/Climate-Change-Awareness-Dashboard.mlflow/#/experiments/10
val skill all rows : 0.0301   (4.7.0 run: 0.0301)
val skill no |z|>6 : 0.0335   (4.7.0 run: 0.0335)
train skill        : 0.2062
reload max abs diff: 0.00e+00   (must be below 1e-9)
model.txt size MB  : 0.96
{'overall': np.float64(0.0301), 'season_DJF': np.float64(-0.0093), 'season_JJA': np.float64(0.0896), 'season_MAM': np.float64(0.0296), 'season_SON': np.float64(0.0031), 

In [4]:
import mlflow.lightgbm

assert lgb.__version__ == "4.7.0", f"Expected 4.7.0, got {lgb.__version__}"

PKG_DIR = Path("../inference/rain_package")
PKG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_TXT = PKG_DIR / "model.txt"

train = df[df["year"].between(TRAIN_START, TRAIN_END)]
val = df[df["year"].between(VAL_START, VAL_END)]

freeze = {
    "params": DEFAULT_PARAMS, "clip_target_z": None, "features": FEATURES,
    "dataset_md5": DATASET_MD5, "git_commit": GIT_COMMIT, "sd_floor_mm": SD_FLOOR,
    "lightgbm_version": lgb.__version__, "train_years": [TRAIN_START, TRAIN_END],
    "selection": "default kept: no CV setting had both folds above 0",
}
with open(PKG_DIR / "freeze_record.json", "w") as f:
    json.dump(freeze, f, indent=2)

mlflow.set_experiment("rain_v2_3month_lightgbm")
with mlflow.start_run(run_name="rain_v2_lgbm_final_b", log_system_metrics=False):
    mlflow.log_params(DEFAULT_PARAMS)
    mlflow.set_tags({"git_commit": GIT_COMMIT, "dataset_md5": DATASET_MD5,
                     "lightgbm_version": lgb.__version__,
                     "season_definition": "origin month",
                     "known_leak": "normals use all train years"})

    t0 = time.time()
    model = fit_lgbm(train[FEATURES], train["target_z"], DEFAULT_PARAMS)
    mlflow.log_metric("fit_seconds", time.time() - t0)

    pred_val = model.predict(val[FEATURES])
    res_all = evaluate(val, pred_val)
    res_noext = evaluate(val, pred_val, drop_extreme=True)
    for k, v in res_all.items():
        mlflow.log_metric(f"val_{k}", v)
    for k, v in res_noext.items():
        mlflow.log_metric(f"val_noext_{k}", v)
    pred_tr = model.predict(train[FEATURES])
    mlflow.log_metric("train_overall", mse_skill(train["target_z"], pred_tr))

    model.booster_.save_model(str(MODEL_TXT))
    booster = lgb.Booster(model_file=str(MODEL_TXT))
    reload_diff = float(np.max(np.abs(booster.predict(val[FEATURES]) - pred_val)))
    mlflow.log_metric("reload_max_abs_diff", reload_diff)

    mlflow.log_artifact(str(MODEL_TXT))
    mlflow.log_artifact(str(PKG_DIR / "freeze_record.json"))
    mlflow.lightgbm.log_model(model.booster_, name="model")
    FINAL_RUN_ID = mlflow.active_run().info.run_id

print(f"val skill all rows : {res_all['overall']:.4f}   (expected 0.0301)")
print(f"val skill no |z|>6 : {res_noext['overall']:.4f}   (expected 0.0335)")
print(f"train skill        : {mse_skill(train['target_z'], pred_tr):.4f}")
print(f"reload max abs diff: {reload_diff:.2e}   (must be below 1e-9)")
print("model.txt size MB  :", round(MODEL_TXT.stat().st_size / 1e6, 2))
print("run id             :", FINAL_RUN_ID)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.155471 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4046
[LightGBM] [Info] Number of data points in the train set: 5980344, number of used features: 23
[LightGBM] [Info] Start training from score 0.000904
🏃 View run rain_v2_lgbm_final_b at: https://dagshub.com/Sharanya-0224/Climate-Change-Awareness-Dashboard.mlflow/#/experiments/10/runs/f36994a1067f4d86bec47a77ec95ecf0
🧪 View experiment at: https://dagshub.com/Sharanya-0224/Climate-Change-Awareness-Dashboard.mlflow/#/experiments/10
val skill all rows : 0.0301   (expected 0.0301)
val skill no |z|>6 : 0.0335   (expected 0.0335)
train skill        : 0.2062
reload max abs diff: 0.00e+00   (must be below 1e-9)
model.txt size MB  : 0.96
run id             : f36994a1067f4d86bec47a77ec95ecf0
